# A股四大主流量化策略演示

数据：baostock 上证50成分股日线（后复权）+ 上证50/沪深300指数。
本 notebook **只做可视化**：所有数据处理与回测逻辑在 `src/quant/` 中，结果由下列命令生成后保存在 `data/results/`：

```bash
uv run python -m quant.data   # 下载数据并构建面板
uv run python -m quant.run    # 运行四个策略，保存结果
```

统一约定：t 日收盘出信号、t+1 日开盘成交；单边费用 0.1% + 滑点 0.1%；回测区间 2017-01 起。

> ⚠️ 股票池为**当前**上证50成分股，历史回测存在幸存者偏差；结果仅用于方法演示，不构成投资建议。

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

RESULTS = Path("..") / "data" / "results"
def load(name: str) -> dict:
    return pd.read_pickle(RESULTS / f"{name}.pkl")

# ---- 图表样式：细线、浅网格、固定的分类色顺序 ----
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BENCH = "#898781"  # 基准统一用中性灰
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
DIVERGING = [[0, "#e34948"], [0.5, "#f0efec"], [1, "#2a78d6"]]  # 负-红 / 0-灰 / 正-蓝
ORDINAL = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]

pio.templates["quant"] = go.layout.Template(layout=dict(
    font=dict(family="system-ui, -apple-system, 'Segoe UI', 'Microsoft YaHei', sans-serif", color=INK2, size=12),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", colorway=SERIES,
    xaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False),
    yaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zerolinecolor="#c3c2b7"),
    legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0),
    hovermode="x unified", margin=dict(l=60, r=20, t=70, b=40), height=420,
))
pio.templates.default = "quant"

PCT_COLS = ["累计收益", "年化收益", "年化波动", "最大回撤", "日胜率"]
def show_stats(stats: pd.DataFrame):
    fmt = {c: "{:.1%}" for c in stats.columns if c in PCT_COLS}
    fmt.update({c: "{:.2f}" for c in stats.columns if c not in PCT_COLS})
    return stats.style.format(fmt) # type: ignore

def color_of(name: str, i: int) -> str:
    return BENCH if ("指数" in name or "买入持有" in name or "等权" in name and i > 1) else SERIES[i]

def plot_equity(res: dict, title: str):
    """净值（对数坐标）+ 回撤，上下两张子图共享时间轴。"""
    eq, dd = res["equity"] / res["equity"].iloc[0], res["drawdown"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.04)
    for i, col in enumerate(eq.columns):
        c = color_of(col, i)
        dash = "dot" if c == BENCH and "指数" in col else None
        fig.add_scatter(x=eq.index, y=eq[col], name=col, line=dict(color=c, width=2, dash=dash), row=1, col=1,
                        legendgroup=col, hovertemplate="%{y:.2f}")
        fig.add_scatter(x=dd.index, y=dd[col], name=col, line=dict(color=c, width=1.5, dash=dash), row=2, col=1,
                        legendgroup=col, showlegend=False, hovertemplate="%{y:.1%}")
    fig.update_yaxes(type="log", title_text="净值（对数）", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", title_text="回撤", row=2, col=1)
    fig.update_layout(title=title, height=600)
    fig.show()

def plot_grid(grid: pd.DataFrame, metric: str, title: str, fmt: str = ".2f"):
    """参数网格热力图（发散色，0 为灰）。"""
    m = grid[metric].unstack()
    lim = max(abs(m.min().min()), abs(m.max().max()))
    fig = go.Figure(go.Heatmap(
        z=m.values, x=[str(c) for c in m.columns], y=[str(i) for i in m.index], colorscale=DIVERGING,
        zmin=-lim, zmax=lim, text=m.map(lambda v: "" if pd.isna(v) else format(v, fmt)).values,
        texttemplate="%{text}", xgap=2, ygap=2, colorbar=dict(title=metric),
        hovertemplate=f"{m.index.name}=%{{y}}<br>{m.columns.name}=%{{x}}<br>{metric}=%{{z:{fmt}}}<extra></extra>",
    ))
    fig.update_layout(title=title, xaxis_title=m.columns.name, yaxis_title=m.index.name, hovermode="closest",
                      height=380, xaxis=dict(type="category", showgrid=False), yaxis=dict(type="category", showgrid=False))
    fig.show()

## 1. 动量策略（横截面动量）

**逻辑**：强者恒强。每月末计算每只股票过去 12 个月、剔除最近 1 个月的收益（12-1 动量），买入排名前 10 的股票等权持有，下月初调仓。同时构建 Bottom10（最弱）组合做对照：如果动量有效，Top 应持续跑赢 Bottom。

代码：`src/quant/strategies/momentum.py`

In [2]:
mom = load("momentum")
print(mom["params"])
show_stats(mom["stats"])

{'lookback': 252, 'skip': 21, 'top_n': 10}


,累计收益,年化收益,年化波动,夏普比率,最大回撤,卡玛比率,日胜率
动量 Top10,737.3%,25.1%,28.1%,0.94,-37.2%,0.67,51.9%
动量 Bottom10,305.1%,15.9%,22.1%,0.78,-32.3%,0.49,49.4%
股票池等权,450.9%,19.7%,18.3%,1.07,-24.1%,0.82,53.0%
上证50指数,20.1%,1.9%,18.1%,0.20,-45.4%,0.04,50.4%


In [3]:
plot_equity(mom, "动量策略：Top10 vs Bottom10 vs 基准")

**参数敏感性**：不同回看期和持仓数下的夏普比率。一个稳健的策略不应只在某个特定参数上有效。

In [4]:
plot_grid(mom["grid"], "夏普比率", "动量策略参数网格：夏普比率")

**换手与持仓**：动量策略月度换手率，以及最新一期持仓。

In [5]:
fig = go.Figure(go.Bar(x=mom["turnover"].index, y=mom["turnover"], marker_color=SERIES[0],
                       hovertemplate="%{x|%Y-%m}<br>换手 %{y:.0%}<extra></extra>"))
fig.update_layout(title=f"月度单边换手率（均值 {mom['turnover'].mean():.0%}）", yaxis_tickformat=".0%", height=320, hovermode="closest")
fig.show()
mom["latest_holdings"].rename("权重").to_frame().style.format("{:.1%}")

,权重
code,
sh.600183 生益科技,10.0%
sh.601088 中国神华,10.0%
sh.601328 交通银行,10.0%
sh.601857 中国石油,10.0%
sh.601919 中远海控,10.0%
sh.601988 中国银行,10.0%
sh.603259 药明康德,10.0%
sh.603986 兆易创新,10.0%
sh.688008 澜起科技,10.0%


## 2. 均值回归策略（布林带）

**逻辑**：价格短期偏离均值过多会回归。计算 Z = (价格 − 20日均线) / 20日标准差，Z 下穿 −2（跌破布林下轨）买入，Z 回到 0（回到均线）卖出，另设 10% 止损。每只股票独立回测、分配等额资金后汇总。

注意：该策略大部分时间空仓（见下表"持仓时间占比"），资金利用率低，所以组合层面收益与波动都偏小——这正是择时类均值回归的典型特征。

代码：`src/quant/strategies/mean_reversion.py`

In [6]:
mr = load("mean_reversion")
print(mr["params"])
show_stats(mr["stats"])

{'window': 20, 'k': 2.0, 'stop_loss': 0.1}


,累计收益,年化收益,年化波动,夏普比率,最大回撤,卡玛比率,日胜率
"均值回归 BB(20,2.0)",26.8%,2.5%,5.7%,0.47,-10.3%,0.25,46.5%
买入持有(等额),271.6%,14.8%,20.1%,0.79,-32.6%,0.45,51.4%
上证50指数,20.1%,1.9%,18.1%,0.20,-45.4%,0.04,50.4%


In [7]:
plot_equity(mr, "均值回归：组合净值 vs 买入持有")

**单只股票示例**：价格、布林带与实际买卖点（取最近两年以便看清）。

In [8]:
ex = mr["example"]
b = ex["bands"].loc[ex["bands"].index[-500]:]
t = ex["trades"]
t = t[t["Entry Timestamp"] >= b.index[0]]
fig = go.Figure()
fig.add_scatter(x=b.index, y=b["upper"], line=dict(color=GRID, width=1), name="上轨", hoverinfo="skip")
fig.add_scatter(x=b.index, y=b["lower"], line=dict(color=GRID, width=1), name="下轨", fill="tonexty",
                fillcolor="rgba(225,224,217,0.35)", hoverinfo="skip")
fig.add_scatter(x=b.index, y=b["ma"], line=dict(color=BENCH, width=1.5, dash="dot"), name="20日均线")
fig.add_scatter(x=b.index, y=b["close"], line=dict(color=SERIES[0], width=2), name="收盘价(后复权)")
fig.add_scatter(x=t["Entry Timestamp"], y=t["Avg Entry Price"], mode="markers", name="买入",
                marker=dict(symbol="triangle-up", size=11, color=SERIES[2], line=dict(color="#fcfcfb", width=2)))
fig.add_scatter(x=t["Exit Timestamp"], y=t["Avg Exit Price"], mode="markers", name="卖出",
                marker=dict(symbol="triangle-down", size=11, color=SERIES[1], line=dict(color="#fcfcfb", width=2)))
fig.update_layout(title=f"{ex['code']} {ex['name']}：布林带均值回归买卖点", height=480)
fig.show()
t[["Entry Timestamp", "Exit Timestamp", "Avg Entry Price", "Avg Exit Price", "Return", "Status"]].style.format(
    {"Return": "{:.1%}", "Avg Entry Price": "{:.1f}", "Avg Exit Price": "{:.1f}"})

,Entry Timestamp,Exit Timestamp,Avg Entry Price,Avg Exit Price,Return,Status
505,2025-01-06 00:00:00,2025-02-13 00:00:00,10508.3,10415.3,-1.1%,Closed
506,2025-06-16 00:00:00,2025-07-07 00:00:00,10133.7,10466.8,3.1%,Closed
507,2025-12-10 00:00:00,2025-12-22 00:00:00,10323.4,10553.1,2.0%,Closed
508,2026-01-22 00:00:00,2026-01-30 00:00:00,10139.3,10740.2,5.7%,Closed
509,2026-06-22 00:00:00,2026-07-13 00:00:00,9106.7,9171.8,0.5%,Closed
510,2026-09-17 00:00:00,2026-10-13 00:00:00,9657.4,9860.5,1.9%,Closed


**逐只股票表现**：策略收益 vs 买入持有收益。

In [9]:
ps = mr["per_stock"].sort_values("策略收益")
labels = ps.index + " " + ps["名称"]
fig = go.Figure()
fig.add_bar(y=labels, x=ps["买入持有收益"], orientation="h", name="买入持有", marker_color=BENCH)
fig.add_bar(y=labels, x=ps["策略收益"], orientation="h", name="均值回归", marker_color=SERIES[0])
fig.update_layout(title="各股票累计收益：均值回归 vs 买入持有", barmode="group", bargap=0.3, bargroupgap=0.1,
                  height=max(500, 18 * len(ps)), xaxis_tickformat=".0%", hovermode="y unified")
fig.show()
ps.style.format({"策略收益": "{:.1%}", "买入持有收益": "{:.1%}", "胜率": "{:.1%}", "持仓时间占比": "{:.1%}", "交易次数": "{:.0f}"})

,名称,策略收益,买入持有收益,交易次数,胜率,持仓时间占比
code,,,,,,
sh.601012,隆基绿能,-62.0%,212.9%,43,39.5%,19.8%
sh.601628,中国人寿,-43.0%,61.0%,41,56.1%,22.6%
sh.601857,中国石油,-34.2%,125.8%,40,52.5%,25.4%
sh.600309,万华化学,-27.0%,372.7%,36,52.8%,17.4%
sh.601318,中国平安,-26.8%,119.9%,36,55.6%,21.9%
sh.600276,恒瑞医药,-14.6%,189.9%,43,62.8%,21.0%
sh.601398,工商银行,-10.6%,227.4%,26,61.5%,16.3%
sh.600036,招商银行,-9.6%,253.2%,35,54.3%,18.6%
sh.601166,兴业银行,-9.3%,96.8%,41,56.1%,23.5%


In [10]:
plot_grid(mr["grid"], "夏普比率", "均值回归参数网格：窗口 × 阈值")

## 3. 趋势跟随策略（沪深300 择时）

**逻辑**：顺势而为——趋势形成后跟进、反转时离场。牺牲震荡市中的小额亏损，换取在大趋势中保住利润、躲开大跌。
- **双均线**：20 日均线上穿 60 日均线买入，下穿卖出
- **唐奇安通道**（海龟法则简化）：收盘突破过去 55 日最高价买入，跌破过去 20 日最低价卖出

趋势策略的价值通常更多体现在**降低回撤**而不是提高收益。

代码：`src/quant/strategies/trend.py`

In [11]:
tr = load("trend")
print(tr["params"])
show_stats(tr["stats"])

{'target': 'sh.000300', 'fast': 20, 'slow': 60, 'entry_n': 55, 'exit_n': 20}


,累计收益,年化收益,年化波动,夏普比率,最大回撤,卡玛比率,日胜率
"双均线 MA(20,60)",-3.5%,-0.4%,12.9%,0.04,-41.1%,-0.01,27.1%
"唐奇安 (55,20)",0.2%,0.0%,11.4%,0.06,-28.3%,0.00,17.6%
沪深300 买入持有,21.0%,2.0%,18.6%,0.20,-45.6%,0.04,50.8%


In [12]:
plot_equity(tr, "趋势跟随：沪深300 择时")

**信号示意**：沪深300 与快慢均线，浅色背景为双均线策略持仓期。

In [13]:
p, pos = tr["price"], tr["position_ma"].reindex(tr["price"].index).fillna(False)
# 把连续持仓区间画成背景色块
blocks = (pos != pos.shift()).cumsum()[pos]
fig = go.Figure()
for _, seg in pos[pos].groupby(blocks):
    fig.add_vrect(x0=seg.index[0], x1=seg.index[-1], fillcolor="rgba(42,120,214,0.10)", line_width=0)
fig.add_scatter(x=p.index, y=p["close"], name="沪深300", line=dict(color=INK2, width=1.5))
fig.add_scatter(x=p.index, y=p["ma_fast"], name=f"MA{tr['params']['fast']}", line=dict(color=SERIES[0], width=2))
fig.add_scatter(x=p.index, y=p["ma_slow"], name=f"MA{tr['params']['slow']}", line=dict(color=SERIES[1], width=2))
fig.update_layout(title="沪深300 双均线信号（蓝色背景 = 持仓）", height=460)
fig.show()

In [14]:
p = tr["price"]
fig = go.Figure()
fig.add_scatter(x=p.index, y=p["dc_upper"], name="55日最高（入场线）", line=dict(color=SERIES[2], width=1.5))
fig.add_scatter(x=p.index, y=p["dc_lower"], name="20日最低（离场线）", line=dict(color=SERIES[1], width=1.5))
fig.add_scatter(x=p.index, y=p["close"], name="沪深300", line=dict(color=INK2, width=1.5))
t = tr["trades_dc"]
fig.add_scatter(x=t["Entry Timestamp"], y=t["Avg Entry Price"], mode="markers", name="买入",
                marker=dict(symbol="triangle-up", size=10, color=SERIES[2], line=dict(color="#fcfcfb", width=2)))
fig.add_scatter(x=t["Exit Timestamp"], y=t["Avg Exit Price"], mode="markers", name="卖出",
                marker=dict(symbol="triangle-down", size=10, color=SERIES[1], line=dict(color="#fcfcfb", width=2)))
fig.update_layout(title="唐奇安通道突破信号", height=460)
fig.show()

**参数网格**：所有 (快线, 慢线) 组合一次性用 vectorbt 向量化回测。看夏普和最大回撤是否在一片区域内都较好，而不是孤立的最优点。

In [15]:
plot_grid(tr["grid"], "夏普比率", "双均线参数网格：夏普比率")
plot_grid(tr["grid"], "最大回撤", "双均线参数网格：最大回撤（越接近 0 越好）", fmt=".0%")

## 4. 因子投资（多因子选股）

**逻辑**：用有经济学解释的因子给股票打分，长期持有高分股票以获取因子溢价。
- 因子：价值 EP、价值 BP、低波动、短期反转、小市值（另有 12-1 动量作对照）
- 处理：横截面去极值 → 标准化 → 等权合成复合因子
- 检验：**Rank IC**（因子与下月收益的秩相关，|IC|>0.03、ICIR>0.3 通常认为有效）、**五分组分层**（Q5 最高分 → Q1 最低分，理想情况单调）、Top10 含成本回测

注意：上证50 只有 50 只股票，每组仅约 10 只，因子检验的统计效力有限；实际研究通常用全市场或中证800等更宽的股票池。

代码：`src/quant/strategies/factor.py`

In [16]:
fac = load("factor")
fac["ic_summary"].style.format({"IC均值": "{:.3f}", "IC标准差": "{:.3f}", "ICIR": "{:.2f}", "t值": "{:.2f}", "IC>0占比": "{:.0%}"})

,IC均值,IC标准差,ICIR,t值,IC>0占比
价值EP,0.038,0.315,0.12,1.33,54%
价值BP,0.009,0.338,0.03,0.29,50%
低波动,0.024,0.338,0.07,0.78,55%
短期反转,-0.008,0.253,-0.03,-0.33,42%
小市值,0.005,0.290,0.02,0.20,50%
动量12-1,0.015,0.273,0.05,0.59,51%
复合因子,0.016,0.278,0.06,0.64,53%


In [17]:
ic_cum = fac["ic"].cumsum()
fig = go.Figure()
for i, col in enumerate(ic_cum.columns):
    is_comp = col == "复合因子"
    fig.add_scatter(x=ic_cum.index, y=ic_cum[col], name=col,
                    line=dict(color=INK if is_comp else SERIES[i], width=3 if is_comp else 1.5))
fig.update_layout(title="累计 Rank IC（持续向上 = 因子方向稳定有效）", height=440)
fig.show()

In [18]:
nav = fac["layer_nav"]
fig = go.Figure()
for i, col in enumerate([c for c in nav.columns if c.startswith("Q")]):
    fig.add_scatter(x=nav.index, y=nav[col], name=col, line=dict(color=ORDINAL[i], width=2))
fig.add_scatter(x=nav.index, y=nav["多空(Q5-Q1)"], name="多空(Q5-Q1)", line=dict(color=SERIES[1], width=2, dash="dash"))
fig.update_layout(title="复合因子五分组净值（未扣费；Q5 = 得分最高）", yaxis_type="log", height=460)
fig.show()

ann = (1 + fac["layer_returns"]).prod() ** (12 / len(fac["layer_returns"])) - 1
fig = go.Figure(go.Bar(x=ann.index, y=ann.values, marker_color=ORDINAL + [SERIES[1]],
                       text=[f"{v:.1%}" for v in ann.values], textposition="outside",
                       hovertemplate="%{x}<br>年化 %{y:.1%}<extra></extra>"))
fig.update_layout(title="各分组年化收益（理想情况：Q1→Q5 单调递增）", yaxis_tickformat=".0%", height=360, hovermode="closest")
fig.show()

In [19]:
c = fac["factor_corr"]
fig = go.Figure(go.Heatmap(z=c.values, x=c.columns, y=c.index, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=c.round(2).values, texttemplate="%{text}", xgap=2, ygap=2))
fig.update_layout(title="因子间平均横截面相关系数（相关性低 → 合成后分散效果好）", height=440, hovermode="closest",
                  xaxis=dict(showgrid=False), yaxis=dict(showgrid=False, autorange="reversed"))
fig.show()

In [20]:
show_stats(fac["stats"])

,累计收益,年化收益,年化波动,夏普比率,最大回撤,卡玛比率,日胜率
复合因子 Top10,188.2%,11.8%,15.5%,0.80,-21.3%,0.55,51.2%
股票池等权,450.9%,19.7%,18.3%,1.07,-24.1%,0.82,53.0%
上证50指数,20.1%,1.9%,18.1%,0.20,-45.4%,0.04,50.4%


In [21]:
plot_equity(fac, "复合因子 Top10（含交易成本）")

In [22]:
print("最新打分日期:", fac["latest_date"].date())
fac["latest_scores"].head(15).style.format({c: "{:.2f}" for c in fac["latest_scores"].columns if c != "名称"})

最新打分日期: 2026-11-12


,名称,价值EP,价值BP,低波动,短期反转,小市值,复合因子
code,,,,,,,
sh.601668,中国建筑,1.70,2.01,1.02,-0.34,0.45,0.97
sh.601688,华泰证券,1.15,1.11,0.78,0.44,0.90,0.88
sh.601328,交通银行,1.44,2.01,0.88,-0.76,0.30,0.77
sh.601600,中国铝业,1.63,-0.06,0.06,1.10,1.02,0.75
sh.601658,邮储银行,1.46,2.01,0.76,-0.13,-0.41,0.74
sh.601211,国泰海通,0.88,1.10,0.56,0.61,0.29,0.69
sh.601166,兴业银行,1.70,2.01,1.02,-1.00,-0.42,0.66
sh.600050,中国联通,-0.50,1.40,0.98,0.61,0.82,0.66
sh.601919,中远海控,1.44,0.62,0.20,0.02,0.32,0.52


## 5. 四种策略汇总

In [23]:
summary = load("summary")["stats"]
show_stats(summary)

In [24]:
eq = pd.concat({
    "动量 Top10": mom["equity"].iloc[:, 0],
    "均值回归": mr["equity"].iloc[:, 0],
    "趋势(双均线)": tr["equity"].iloc[:, 0],
    "复合因子 Top10": fac["equity"].iloc[:, 0],
    "上证50指数": mom["equity"]["上证50指数"],
}, axis=1).dropna()
eq = eq / eq.iloc[0]
fig = go.Figure()
for i, col in enumerate(eq.columns):
    bench = col == "上证50指数"
    fig.add_scatter(x=eq.index, y=eq[col], name=col, line=dict(color=BENCH if bench else SERIES[i], width=2, dash="dot" if bench else None))
fig.update_layout(title="四种策略净值对比（对数坐标）", yaxis_type="log", height=480)
fig.show()

### 小结

| 策略 | 类型 | 赚的是什么钱 | 主要风险 |
|---|---|---|---|
| 动量 | 横截面选股 | 投资者对信息反应不足、羊群效应 | 风格切换时的"动量崩溃"；A股短期反转效应较强 |
| 均值回归 | 单标的择时 | 短期过度反应后的修正 | 趋势性下跌中"接飞刀"，资金利用率低 |
| 趋势跟随 | 指数择时 | 大级别趋势的延续 | 震荡市反复止损（胜率低、靠少数大盈利） |
| 因子投资 | 横截面选股 | 风险补偿/行为偏差带来的因子溢价 | 因子拥挤与失效、风格长期回撤 |

局限：幸存者偏差（当前成分股）、未考虑涨跌停无法成交、停牌日不补单、股票池较小。